# 04 — RG-improved Schwarzschild geometry

Bonanno–Reuter (2000): promote Newton's constant to G(r) by
evaluating an RG trajectory at scale k(r). The improved metric
weakens gravity at short distances (G(r) → 0 at the origin) and has
a critical mass below which no horizon forms. With the k = ξ/r
identification used here the central singularity is *softened*
(f(0) = 1 but R ~ 1/r), not removed — Bonanno–Reuter's fully regular
de Sitter core needs the softened distance scale k ~ 1/d(r) with
d(r) ~ r^{3/2}, which is not implemented.


In [ ]:
import numpy as np
from asymsafety.beta.einstein_hilbert import build_eh_beta_system
from asymsafety.analysis.fixed_points import FixedPointFinder
from asymsafety.analysis.flow import FlowIntegrator
from asymsafety.cosmology.scale_identification import InverseDistanceScale
from asymsafety.cosmology.rg_improved_bh import RGImprovedSchwarzschild
from asymsafety.cosmology.visualization import (
    plot_running_newton_constant, plot_lapse_with_horizons,
    plot_classical_vs_rg_lapse, plot_hawking_temperature,
)


## 1. Build a UV→IR trajectory

Start from just inside the basin of the NGFP and flow toward small t (IR).


In [ ]:
system = build_eh_beta_system(d=4)
ngfp = FixedPointFinder(system).find_fixed_point({'g': 0.7, 'lambda': 0.14})
ic_uv = {'g': ngfp.location['g'] - 0.001, 'lambda': ngfp.location['lambda'] + 0.001}
traj = FlowIntegrator(system).integrate(ic_uv, t_span=(10, -10), max_step=0.05)
print(f'Trajectory has {len(traj.t_values)} points.')


## 2. RG-improved geometry for an O(1) mass


In [ ]:
bh = RGImprovedSchwarzschild(
    traj, scale=InverseDistanceScale(xi=1.0), M=1.0, k0=1.0,
)
fig_G = plot_running_newton_constant(bh, r_range=(1e-3, 100.0))
fig_f = plot_lapse_with_horizons(bh, r_range=(1e-3, 100.0))
fig_cv = plot_classical_vs_rg_lapse(bh, M_values=(0.4, 1.0, 2.5))
fig_T = plot_hawking_temperature(bh, M_range=(0.3, 4.0), n_masses=40)


## 3. Critical mass — no horizon below it


In [ ]:
M_crit = bh.critical_mass(M_search=(1e-4, 5.0))
print(f'Critical mass M_crit ≈ {M_crit:.4e} (in units k0=1)')

for M in [1e-4, M_crit*0.5, M_crit*2.0, 1.0, 10.0]:
    bhM = RGImprovedSchwarzschild(traj, M=M, k0=1.0)
    h = bhM.horizons(1e-3, 1000, 5000)
    label = f'M = {M:.3e}'
    print(f'  {label:25s}: {len(h)} horizon(s) at {h}')


**Key takeaways:**

- `G(r)` falls to zero at the origin, weakening gravity at short distances.
- The lapse `f(r)` returns to 1 at the origin, but only linearly (`f'(0) ≠ 0`):
  the curvature still diverges, `R ≈ 12 g* M/r` — a much milder singularity
  than classical Schwarzschild, not its removal. (A true de Sitter core,
  `1 - f ∝ r²` with finite curvature, would require the Bonanno–Reuter
  softened scale `k ~ 1/d(r)`, `d(r) ~ r^{3/2}`, not the plain `k = ξ/r`.)
- For `M < M_crit`, the gravitational well is too shallow to form a horizon.
